In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import mne
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')
from scipy.signal import filtfilt
from scipy.spatial.distance import pdist, squareform, cdist
from scipy.linalg import pinv
from scipy.stats import pearsonr, spearmanr, f_oneway
import rsatoolbox
from rsatoolbox.data import Dataset as RSADataset
from rsatoolbox.data import TemporalDataset
from rsatoolbox.rdm import calc_rdm
from rsatoolbox.rdm.calc import calc_rdm_movie
from joblib import Parallel, delayed
from tqdm import tqdm
from mne.stats import permutation_cluster_1samp_test, permutation_cluster_test, combine_adjacency


# Import plot config
from plot_config import set_plot_style
set_plot_style()


In [ ]:
# import numpy, rsatoolbox
# data = rsatoolbox.data.Dataset(numpy.random.rand(10, 5))
# rdms = rsatoolbox.rdm.calc_rdm(data)
# rsatoolbox.vis.show_rdm(rdms)


In [ ]:
# Setup paths
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load demographics data
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Filter out Iowa patients
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

print(f"Available subjects: {subj_df.MSSMCode.values}")

# Select one test subject (UI006 if available, otherwise first)
if 'UI006' in subj_df.MSSMCode.values:
    test_subject = 'UI006'
else:
    test_subject = subj_df.MSSMCode.values[0]

print(f"Selected test subject: {test_subject}")


In [ ]:
# Load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

print(f"Loaded learn_df: {learn_df.shape}")
print(f"Loaded combined_df: {combined_df.shape}")

# Filter to test subject
subj_learn_df = learn_df[learn_df.participant == test_subject].copy()
subj_combined_df = combined_df[combined_df.participant == test_subject].copy()

print(f"Subject {test_subject} learn trials: {len(subj_learn_df)}")
print(f"Subject {test_subject} combined trials: {len(subj_combined_df)}")


In [ ]:
# =============================================================================
# EFFICIENT RSA FUNCTIONS
# Data is loaded ONCE, then only RSA computation is repeated for permutations
# =============================================================================

def load_subject_data(subject, base_dir, load_dir, feedback_roi='OFC', cue_roi='HPC',
                      downsample_factor=5, win_ms=200, hfa_freq_range=(70., 201.), 
                      min_trials=10, verbose=True):
    """
    Load and preprocess all data for a single subject (done ONCE before permutations).
    
    Returns dict with: feedback_hfa, cue_hfa, memory_outcomes, feedback_times, cue_times, etc.
    Returns None if subject cannot be analyzed.
    """
    if verbose:
        print(f"\n{'='*50}")
        print(f"LOADING DATA: {subject}")
        print(f"Feedback ROI: {feedback_roi}, Cue ROI: {cue_roi}")
        print(f"{'='*50}")
    
    # Load behavioral data
    try:
        learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
        combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')
    except FileNotFoundError as e:
        if verbose:
            print(f"Behavioral data not found: {e}")
        return None
    
    subj_learn_df = learn_df[learn_df.participant == subject].copy()
    subj_combined_df = combined_df[(combined_df.condition == 'Day1') & 
                                    (combined_df.participant == subject)].dropna(subset=['trials_gamble']).copy()
    
    if len(subj_learn_df) == 0 or len(subj_combined_df) == 0:
        if verbose:
            print(f"No data found for subject {subject}")
        return None
    
    # Establish the trial mapping between encoding and retrieval
    encoding_trials = (subj_combined_df['trials_gamble'].values - 1).astype(int)
    retrieval_trials = (subj_combined_df['trials_mem'].values - 1).astype(int)
    
    # # Remove trials with fast reaction times (< 0.3s)
    # trials_to_remove = (subj_combined_df.loc[(subj_combined_df.recog_rt < 0.3), 
    #                                           'trials_gamble'].values - 1).astype(int)
    # mask = np.isin(encoding_trials, trials_to_remove, invert=True)
    # encoding_trials = encoding_trials[mask]
    # retrieval_trials = retrieval_trials[mask]
    # subj_combined_df = subj_combined_df[mask].copy()
    
    # Load electrode data
    elec_file = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subject}/Day1_reref_elec_df'
    if not os.path.exists(elec_file):
        if verbose:
            print(f"Electrode file not found for subject {subject}")
        return None
    
    elec_df = pd.read_csv(elec_file)
    
    # Get ROI electrodes
    feedback_roi_electrodes = elec_df[elec_df.salman_region == feedback_roi]['label'].tolist()
    cue_roi_electrodes = elec_df[elec_df.salman_region == cue_roi]['label'].tolist()
    
    if len(feedback_roi_electrodes) == 0 or len(cue_roi_electrodes) == 0:
        if verbose:
            print(f"No electrodes found in target ROI for subject {subject}")
        return None
    
    # Load TFR data
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subject}/scratch/TFR'
    feedback_tfr_file = f'{filepath}/feedback_start-tfr.h5'
    cue_tfr_file = f'{filepath}/cue_on-tfr.h5'
    
    if not (os.path.exists(feedback_tfr_file) and os.path.exists(cue_tfr_file)):
        if verbose:
            print(f"TFR files not found for subject {subject}")
        return None
    
    feedback_tfr = mne.time_frequency.read_tfrs(feedback_tfr_file)
    cue_tfr = mne.time_frequency.read_tfrs(cue_tfr_file)
    
    # Limit TFRs to relevant channels
    feedback_tfr = feedback_tfr.pick(feedback_roi_electrodes)
    cue_tfr = cue_tfr.pick(cue_roi_electrodes)
    
    # Extract HFA signal
    hfa_mask = (feedback_tfr.freqs >= hfa_freq_range[0]) & (feedback_tfr.freqs <= hfa_freq_range[1])
    
    # Extract HFA for all electrodes: (n_trials, n_electrodes, n_timepoints)
    feedback_hfa = np.nanmean(feedback_tfr.data[:, :, hfa_mask, :], axis=2)
    cue_hfa = np.nanmean(cue_tfr.data[:, :, hfa_mask, :], axis=2)
    
    # Smooth HFA
    win_size = int(feedback_tfr.info['sfreq'] * (win_ms / 1000))
    window = np.ones(win_size) / win_size
    
    feedback_hfa_smooth = np.zeros_like(feedback_hfa)
    cue_hfa_smooth = np.zeros_like(cue_hfa)
    
    for electrode_idx in range(feedback_hfa.shape[1]):
        feedback_hfa_smooth[:, electrode_idx, :] = filtfilt(window, [1], feedback_hfa[:, electrode_idx, :])
    
    for electrode_idx in range(cue_hfa.shape[1]):
        cue_hfa_smooth[:, electrode_idx, :] = filtfilt(window, [1], cue_hfa[:, electrode_idx, :])
    
    # Select trials for encoding and retrieval
    feedback_hfa_smooth = feedback_hfa_smooth[encoding_trials, :, :]
    cue_hfa_smooth = cue_hfa_smooth[retrieval_trials, :, :]
    
    # Downsample
    feedback_hfa_downsampled = feedback_hfa_smooth[:, :, ::int(downsample_factor)]
    cue_hfa_downsampled = cue_hfa_smooth[:, :, ::int(downsample_factor)]
    
    # Get time vectors
    if hasattr(feedback_tfr, 'times'):
        feedback_times = feedback_tfr.times[::int(downsample_factor)]
    else:
        feedback_times = np.arange(feedback_hfa_downsampled.shape[2])
    
    if hasattr(cue_tfr, 'times'):
        cue_times = cue_tfr.times[::int(downsample_factor)]
    else:
        cue_times = np.arange(cue_hfa_downsampled.shape[2])
    
    # Get dimensions
    n_times_feedback = feedback_hfa_downsampled.shape[2]
    n_times_cue = cue_hfa_downsampled.shape[2]
    n_trials = feedback_hfa_downsampled.shape[0]
    
    if n_trials < min_trials:
        if verbose:
            print(f"Insufficient trials ({n_trials} < {min_trials})")
        return None
    
    # Get memory outcomes
    memory_outcomes = np.array(['remembered' if x else 'forgotten' 
                                for x in subj_combined_df.hits.values])
    
    # Get RPE category for crossnobis cross-validation (if available)
    # This should be a categorical variable grouping trials by RPE type
    # Process behavioral data for one subject
    # Extract RPE values
    rpe_values = subj_learn_df['rpe'].values

    # Compute rpe_category using tercile split
    terciles = np.percentile(rpe_values, [33.33, 66.67])
    print(f"RPE terciles: {terciles}")

    rpe_category = []
    for rpe in rpe_values:
        if rpe <= terciles[0]:
            rpe_category.append('negative')
        elif rpe <= terciles[1]:
            rpe_category.append('neutral')
        else:
            rpe_category.append('pos')

    subj_learn_df['rpe_category'] = rpe_category

    # Check distribution
    print(f"\nRPE category distribution:")
    print(subj_learn_df['rpe_category'].value_counts())

    # Add RPE category to combined_df
    subj_combined_df['rpe_category'] = np.nan
    for idx, row in subj_learn_df.iterrows():
        mask = (subj_combined_df.img_path == row.img_path) & (subj_combined_df.participant == row.participant)
        subj_combined_df.loc[mask, 'rpe_category'] = row.rpe_category

    rpe_category = subj_combined_df.rpe_category.dropna().values
    
    if verbose:
        print(f"Data loaded: {feedback_hfa_downsampled.shape} feedback, {cue_hfa_downsampled.shape} cue")
        print(f"Trials: {n_trials} (remembered: {np.sum(memory_outcomes == 'remembered')}, "
              f"forgotten: {np.sum(memory_outcomes == 'forgotten')})")
        if rpe_category is not None:
            unique_cats = np.unique(rpe_category)
            print(f"RPE categories: {unique_cats} (counts: {[np.sum(rpe_category == c) for c in unique_cats]})")
    
    return {
        'feedback_hfa': feedback_hfa_downsampled,
        'cue_hfa': cue_hfa_downsampled,
        'memory_outcomes': memory_outcomes,
        'rpe_category': rpe_category,
        'feedback_times': feedback_times,
        'cue_times': cue_times,
        'n_trials': n_trials,
        'subject': subject,
        'feedback_roi': feedback_roi,
        'cue_roi': cue_roi
    }


def compute_rsa(data, shuffle=False, shuffle_type='trial_labels', random_state=None,
                rdm_method='mahalanobis', rdm_compare_method='spearman'):
    """
    Compute RSA from preprocessed data. This is the ONLY part repeated in permutations.
    
    Parameters
    ----------
    data : dict
        Preprocessed data dictionary from load_subject_data()
    shuffle : bool
        If True, shuffle labels to create null distribution
    shuffle_type : str
        'trial_labels' or 'trial_pairing'
    random_state : int or None
        Random seed for reproducibility
    rdm_method : str
        Method for computing RDMs. Options: 'euclidean', 'correlation', 'mahalanobis', 'crossnobis'
        Note: 'crossnobis' requires rpe_category in data for cross-validation
    rdm_compare_method : str
        Method for comparing RDMs. Options: 'spearman', 'corr', 'cosine', 'rho-a', 'tau-a'
    
    Returns
    -------
    dict with category_heatmaps, diff_heatmap, n_trials_per_category
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    feedback_hfa = data['feedback_hfa']
    cue_hfa = data['cue_hfa'].copy()  # Copy in case we shuffle
    memory_outcomes = data['memory_outcomes'].copy()
    feedback_times = data['feedback_times']
    cue_times = data['cue_times']
    
    # Get rpe_category for crossnobis (if available)
    rpe_category = data.get('rpe_category', None)
    if rpe_category is not None:
        rpe_category = rpe_category.copy()
    
    n_times_feedback = feedback_hfa.shape[2]
    n_times_cue = cue_hfa.shape[2]
    n_trials = feedback_hfa.shape[0]
    
    # Apply shuffling if requested
    if shuffle:
        if shuffle_type == 'trial_labels':
            memory_outcomes = np.random.permutation(memory_outcomes)
        elif shuffle_type == 'trial_pairing':
            cue_hfa = cue_hfa[np.random.permutation(n_trials), :, :]
    
    # Compute per-category RSA heatmaps
    category_heatmaps = {}
    n_trials_per_category = {}
    
    for cat in ['remembered', 'forgotten']:
        cat_mask = (memory_outcomes == cat)
        n_cat_trials = np.sum(cat_mask)
        n_trials_per_category[cat] = n_cat_trials
        
        if n_cat_trials < 3:
            category_heatmaps[cat] = np.full((n_times_feedback, n_times_cue), np.nan)
            continue
        
        # Extract HFA for this category's trials only
        cat_feedback_hfa = feedback_hfa[cat_mask]
        cat_cue_hfa = cue_hfa[cat_mask]
        
        # Create TemporalDataset and compute RDMs
        # For crossnobis, pass obs_descriptors to TemporalDataset and cv_descriptor to calc_rdm_movie
        if rdm_method == 'crossnobis':
            if rpe_category is None:
                raise ValueError(
                    "crossnobis requires rpe_category in data for cross-validation. "
                    "Make sure rpe_category column exists in learn_df."
                )
            
            # Get rpe_category for this memory category subset
            cat_rpe_category = rpe_category[cat_mask]
            
            # Create TemporalDataset with obs_descriptors for crossnobis
            cat_feedback_tempdata = TemporalDataset(
                cat_feedback_hfa, 
                time_descriptors={'time': feedback_times},
                obs_descriptors={'rpe_category': cat_rpe_category}
            )
            cat_cue_tempdata = TemporalDataset(
                cat_cue_hfa, 
                time_descriptors={'time': cue_times},
                obs_descriptors={'rpe_category': cat_rpe_category}
            )
            
            # Use calc_rdm_movie with cv_descriptor
            cat_feedback_rdms = calc_rdm_movie(cat_feedback_tempdata, method=rdm_method, cv_descriptor='rpe_category')
            cat_cue_rdms = calc_rdm_movie(cat_cue_tempdata, method=rdm_method, cv_descriptor='rpe_category')
        else:
            cat_feedback_tempdata = TemporalDataset(cat_feedback_hfa, time_descriptors={'time': feedback_times})
            cat_cue_tempdata = TemporalDataset(cat_cue_hfa, time_descriptors={'time': cue_times})
            
            cat_feedback_rdms = calc_rdm_movie(cat_feedback_tempdata, method=rdm_method)
            cat_cue_rdms = calc_rdm_movie(cat_cue_tempdata, method=rdm_method)
        
        # Compute cross-temporal heatmap
        heatmap = np.zeros((n_times_feedback, n_times_cue))
        for t_fb in range(n_times_feedback):
            for t_cue in range(n_times_cue):
                result = rsatoolbox.rdm.compare(cat_feedback_rdms[t_fb], cat_cue_rdms[t_cue], method=rdm_compare_method)
                heatmap[t_fb, t_cue] = result[0][0]
        
        category_heatmaps[cat] = heatmap
    
    diff_heatmap = category_heatmaps['remembered'] - category_heatmaps['forgotten']
    
    return {
        'category_heatmaps': category_heatmaps,
        'diff_heatmap': diff_heatmap,
        'n_trials_per_category': n_trials_per_category
    }


def analyze_subject_rsa(subject, base_dir, load_dir, feedback_roi='OFC', cue_roi='HPC',
                        downsample_factor=5, win_ms=100, hfa_freq_range=(70., 201.), 
                        min_trials=10, rdm_method='mahalanobis', rdm_compare_method='spearman',
                        verbose=True):
    """
    Convenience function: Load data and compute RSA for a single subject (no permutation).
    
    Parameters
    ----------
    rdm_method : str
        Method for computing RDMs. Options: 'euclidean', 'correlation', 'mahalanobis', 'crossnobis'
    rdm_compare_method : str
        Method for comparing RDMs. Options: 'spearman', 'corr', 'cosine', 'rho-a', 'tau-a'
    """
    data = load_subject_data(subject, base_dir, load_dir, feedback_roi, cue_roi,
                             downsample_factor, win_ms, hfa_freq_range, min_trials, verbose)
    if data is None:
        return None
    
    if verbose:
        print(f"Computing RSA (RDM: {rdm_method}, compare: {rdm_compare_method})...")
    rsa_result = compute_rsa(data, shuffle=False, rdm_method=rdm_method, rdm_compare_method=rdm_compare_method)
    
    return {
        'subject': subject,
        'feedback_roi': feedback_roi,
        'cue_roi': cue_roi,
        'category_heatmaps': rsa_result['category_heatmaps'],
        'diff_heatmap': rsa_result['diff_heatmap'],
        'feedback_times': data['feedback_times'],
        'cue_times': data['cue_times'],
        'n_trials': data['n_trials'],
        'n_trials_per_category': rsa_result['n_trials_per_category'],
        'shuffled': False,
        'rdm_method': rdm_method,
        'rdm_compare_method': rdm_compare_method
    }


def run_permutation_rsa(subject, base_dir, load_dir, feedback_roi='OFC', cue_roi='HPC',
                        n_permutations=1000, n_jobs=-1, shuffle_type='trial_labels',
                        random_seed=42, rdm_method='mahalanobis', rdm_compare_method='spearman',
                        verbose=True, **kwargs):
    """
    Run permutation testing. Data loaded ONCE, only RSA repeated for permutations.
    
    This is the EFFICIENT version - data loading/preprocessing happens once,
    then only compute_rsa() is called in parallel for each permutation.
    
    If n_permutations=0, returns only observed RSA values (no z-scores or p-values).
    
    Parameters
    ----------
    rdm_method : str
        Method for computing RDMs. Options: 'euclidean', 'correlation', 'mahalanobis', 'crossnobis'
    rdm_compare_method : str
        Method for comparing RDMs. Options: 'spearman', 'corr', 'cosine', 'rho-a', 'tau-a'
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"PERMUTATION RSA: {subject}")
        print(f"ROIs: {feedback_roi} → {cue_roi}, Permutations: {n_permutations}")
        print(f"RDM method: {rdm_method}, Compare method: {rdm_compare_method}")
        print(f"{'='*60}")
    
    # STEP 1: Load data ONCE
    if verbose:
        print("\nStep 1: Loading data (done ONCE)...")
    data = load_subject_data(subject, base_dir, load_dir, feedback_roi, cue_roi, 
                             verbose=verbose, **kwargs)
    
    if data is None:
        if verbose:
            print("Data loading failed")
        return None
    
    # STEP 2: Compute observed RSA
    if verbose:
        print("\nStep 2: Computing observed RSA...")
    observed_rsa = compute_rsa(data, shuffle=False, rdm_method=rdm_method, rdm_compare_method=rdm_compare_method)
    
    observed_result = {
        'subject': subject,
        'feedback_roi': feedback_roi,
        'cue_roi': cue_roi,
        'category_heatmaps': observed_rsa['category_heatmaps'],
        'diff_heatmap': observed_rsa['diff_heatmap'],
        'feedback_times': data['feedback_times'],
        'cue_times': data['cue_times'],
        'n_trials': data['n_trials'],
        'n_trials_per_category': observed_rsa['n_trials_per_category'],
        'shuffled': False,
        'rdm_method': rdm_method,
        'rdm_compare_method': rdm_compare_method
    }
    
    # If n_permutations=0, return only observed results (no z-scores)
    if n_permutations == 0:
        if verbose:
            observed_diff = observed_result['diff_heatmap']
            print(f"\nResults summary (no permutations):")
            print(f"  Observed diff range: {np.nanmin(observed_diff):.3f} to {np.nanmax(observed_diff):.3f}")
        
        return {
            'observed': observed_result,
            'null_distribution': None,
            'p_values': None,
            'z_scores': None,
            'n_permutations': 0,
            'null_mean': None,
            'null_std': None,
            'rdm_method': rdm_method,
            'rdm_compare_method': rdm_compare_method
        }
    
    # STEP 3: Run permutations (ONLY RSA computation, NOT data loading)
    if verbose:
        print(f"\nStep 3: Running {n_permutations} permutations (RSA only, data already loaded)...")
    
    rng = np.random.default_rng(random_seed)
    perm_seeds = rng.integers(0, 2**31 - 1, size=n_permutations)
    
    # Parallel execution - only compute_rsa is called, NOT data loading
    perm_results = Parallel(n_jobs=n_jobs, verbose=10 if verbose else 0)(
        delayed(compute_rsa)(data, shuffle=True, shuffle_type=shuffle_type, random_state=seed,
                             rdm_method=rdm_method, rdm_compare_method=rdm_compare_method)
        for seed in perm_seeds
    )
    
    null_distribution = np.stack([r['diff_heatmap'] for r in perm_results], axis=0)
    
    # Compute statistics (vectorized)
    observed_diff = observed_result['diff_heatmap']
    p_values = np.mean(np.abs(null_distribution) >= np.abs(observed_diff), axis=0)
    
    null_mean = np.mean(null_distribution, axis=0)
    null_std = np.std(null_distribution, axis=0)
    null_std[null_std == 0] = np.nan  # Avoid division by zero
    z_scores = (observed_diff - null_mean) / null_std
    
    if verbose:
        print(f"\nResults summary:")
        print(f"  Observed diff range: {np.nanmin(observed_diff):.3f} to {np.nanmax(observed_diff):.3f}")
        print(f"  Z-score range: {np.nanmin(z_scores):.3f} to {np.nanmax(z_scores):.3f}")
        n_sig = np.sum(p_values < 0.05)
        print(f"  Significant timepoints (p < 0.05): {n_sig}/{p_values.size}")
    
    return {
        'observed': observed_result,
        'null_distribution': null_distribution,
        'p_values': p_values,
        'z_scores': z_scores,
        'n_permutations': n_permutations,
        'null_mean': null_mean,
        'null_std': null_std,
        'rdm_method': rdm_method,
        'rdm_compare_method': rdm_compare_method
    }


def run_group_permutation_rsa(subjects, base_dir, load_dir, feedback_roi='OFC', cue_roi='HPC',
                               n_permutations=1000, n_jobs=-1, shuffle_type='trial_labels', 
                               random_seed=42, rdm_method='mahalanobis', rdm_compare_method='spearman',
                               verbose=True, **kwargs):
    """
    Run permutation RSA across multiple subjects. Each subject's data is loaded once.
    
    If n_permutations=0, returns raw RSA values instead of z-scores.
    
    Parameters
    ----------
    rdm_method : str
        Method for computing RDMs. Options: 'euclidean', 'correlation', 'mahalanobis', 'crossnobis'
    rdm_compare_method : str
        Method for comparing RDMs. Options: 'spearman', 'corr', 'cosine', 'rho-a', 'tau-a'
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"GROUP PERMUTATION RSA")
        print(f"Subjects: {len(subjects)}, Permutations/subject: {n_permutations}")
        print(f"RDM method: {rdm_method}, Compare method: {rdm_compare_method}")
        print(f"{'='*70}")
    
    successful_results = []
    successful_subjects = []
    failed_subjects = []
    
    for i, subject in enumerate(subjects):
        if verbose:
            print(f"\n[{i+1}/{len(subjects)}] {subject}")
        
        result = run_permutation_rsa(
            subject, base_dir, load_dir, feedback_roi, cue_roi,
            n_permutations, n_jobs, shuffle_type,
            random_seed + i if random_seed else None,
            rdm_method=rdm_method, rdm_compare_method=rdm_compare_method,
            verbose=False, **kwargs
        )
        
        if result is not None:
            successful_results.append(result)
            successful_subjects.append(subject)
            if verbose:
                print(f"  ✓ Done")
        else:
            failed_subjects.append(subject)
            if verbose:
                print(f"  ✗ Failed")
    
    if not successful_results:
        return None
    
    # Aggregate observed differences (always available)
    observed_diffs = np.stack([r['observed']['diff_heatmap'] for r in successful_results], axis=0)
    group_observed_diff = np.nanmean(observed_diffs, axis=0)
    
    # Handle z-scores: only aggregate if permutations were run
    if n_permutations > 0:
        z_scores_stack = np.stack([r['z_scores'] for r in successful_results], axis=0)
        group_z_scores = np.nanmean(z_scores_stack, axis=0)
    else:
        # No permutations: use raw RSA values instead of z-scores
        z_scores_stack = None
        group_z_scores = None
    
    # Group t-test (always computed on observed differences)
    from scipy import stats
    group_t_stats = np.zeros_like(group_observed_diff)
    group_p_values = np.ones_like(group_observed_diff)
    
    for i in range(group_observed_diff.shape[0]):
        for j in range(group_observed_diff.shape[1]):
            vals = observed_diffs[:, i, j]
            valid = vals[~np.isnan(vals)]
            if len(valid) > 1:
                group_t_stats[i, j], group_p_values[i, j] = stats.ttest_1samp(valid, 0)
    
    if verbose:
        print(f"\nGroup: {len(successful_subjects)}/{len(subjects)} subjects, "
              f"significant: {np.sum(group_p_values < 0.05)}")
    
    feedback_times = successful_results[0]['observed']['feedback_times']
    cue_times = successful_results[0]['observed']['cue_times']
    
    return {
        'subject_results': successful_results,
        'group_observed_diff': group_observed_diff,
        'group_z_scores': group_z_scores,
        'group_t_stats': group_t_stats,
        'group_p_values': group_p_values,
        'observed_diffs_stack': observed_diffs,
        'z_scores_stack': z_scores_stack,
        'feedback_times': feedback_times,
        'cue_times': cue_times,
        'successful_subjects': successful_subjects,
        'failed_subjects': failed_subjects,
        'n_subjects': len(successful_subjects),
        'n_permutations': n_permutations,
        'feedback_roi': feedback_roi,
        'cue_roi': cue_roi,
        'rdm_method': rdm_method,
        'rdm_compare_method': rdm_compare_method
    } 

In [ ]:
# # =============================================================================
# # Example: Single subject analysis (no permutation)
# # =============================================================================
# result = analyze_subject_rsa(
#     subject='MS035',
#     base_dir=base_dir,
#     load_dir=load_dir,
#     feedback_roi='OFC',
#     cue_roi='HPC',
#     downsample_factor=5,
#     verbose=True
# )

# # Plot the result
# if result is not None:
#     fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
#     for idx, (cat, title) in enumerate([('remembered', 'Remembered'), ('forgotten', 'Forgotten')]):
#         ax = axes[idx]
#         im = ax.imshow(result['category_heatmaps'][cat], aspect='auto', origin='lower',
#                        extent=[result['cue_times'][0], result['cue_times'][-1], 
#                               result['feedback_times'][0], result['feedback_times'][-1]],
#                        cmap='RdBu_r')
#         ax.set_title(f"{title} (n={result['n_trials_per_category'][cat]})")
#         ax.set_xlabel('Cue Time (s)')
#         ax.set_ylabel('Feedback Time (s)' if idx == 0 else '')
#         plt.colorbar(im, ax=ax)
    
#     ax = axes[2]
#     vmax = np.nanmax(np.abs(result['diff_heatmap']))
#     im = ax.imshow(result['diff_heatmap'], aspect='auto', origin='lower',
#                    extent=[result['cue_times'][0], result['cue_times'][-1], 
#                           result['feedback_times'][0], result['feedback_times'][-1]],
#                    cmap='RdBu_r', vmin=-vmax, vmax=vmax)
#     ax.set_title('Remembered - Forgotten')
#     ax.set_xlabel('Cue Time (s)')
#     plt.colorbar(im, ax=ax, label='ΔRSA')
    
#     plt.suptitle(f"{result['subject']} | {result['feedback_roi']} → {result['cue_roi']}")
#     plt.tight_layout()
#     plt.show()


In [ ]:
# =============================================================================
# Example: EFFICIENT Permutation Testing (data loaded ONCE)
# # =============================================================================
# # This is the key improvement: data is loaded and preprocessed ONCE, 
# # then only the RSA computation is repeated for each permutation

# perm_result = run_permutation_rsa(
#     # subject='MS035',
#     base_dir=base_dir,
#     load_dir=load_dir,
#     feedback_roi='dmPFC',
#     cue_roi='dmPFC',
#     n_permutations=100,  # Use 1000+ for publication-quality stats
#     n_jobs=8,            # Adjust based on your available cores
#     shuffle_type='trial_labels',  # or 'trial_pairing'
#     random_seed=42,
#     downsample_factor=5,
#     verbose=True
# )


In [ ]:
# =============================================================================
# Run Group Permutation RSA across ALL subjects for MULTIPLE FEEDBACK ROIs
# =============================================================================
import pickle

all_subjects = subj_df.MSSMCode.values.tolist()
print(f"Running analysis on {len(all_subjects)} subjects: {all_subjects}")

# Define ROIs to analyze
feedback_rois = ['ACC', 'dmPFC', 'dlPFC', 'OFC']
cue_roi = 'HPC'

# Directory to save results
save_dir = f'{base_dir}/scratch/MemoryBandit/RSA/crossregion'
os.makedirs(save_dir, exist_ok=True)

# Dictionary to store all results
all_group_results = {}

for feedback_roi in feedback_rois:
    print(f"\n{'='*70}")
    print(f"ANALYZING: {feedback_roi} → {cue_roi}")
    print(f"{'='*70}")
    
    group_results = run_group_permutation_rsa(
        subjects=all_subjects,
        base_dir=base_dir,
        load_dir=load_dir,
        feedback_roi=feedback_roi,
        cue_roi=cue_roi,
        n_permutations=0,
        n_jobs=8,
        shuffle_type='trial_labels',
        random_seed=42,
        downsample_factor=5,
        verbose=True,
        rdm_method='euclidean',
        rdm_compare_method='rho-a'
    )
    
    if group_results is not None:
        # Store in dictionary
        all_group_results[feedback_roi] = group_results
        
        # Save individual result
        save_path = f'{save_dir}/group_results_{feedback_roi}_to_{cue_roi}.pkl'
        with open(save_path, 'wb') as f:
            pickle.dump(group_results, f)
        print(f"Saved: {save_path}")
    else:
        print(f"WARNING: No results for {feedback_roi} → {cue_roi}")

print(f"\n{'='*70}")
print(f"COMPLETE: Analyzed {len(all_group_results)} ROI pairs")
print(f"Results saved to: {save_dir}")
print(f"{'='*70}")


In [ ]:
# Define ROIs to analyze
feedback_rois = ['ACC']
cue_roi = 'HPC'

In [ ]:
# =============================================================================
# MNE Permutation Cluster Test on Group Data - ALL ROIs
# =============================================================================
# Uses the observed_diffs_stack (subjects × feedback_times × cue_times) to test
# for significant clusters where remembered-forgotten differs from 0
# Loops through all saved ROI results

import pickle
from scipy.sparse import csr_matrix, kron, eye

# Load results if not already in memory
save_dir = f'{base_dir}/scratch/MemoryBandit/RSA/crossregion'
feedback_rois = ['ACC', 'dmPFC', 'dlPFC', 'OFC']
cue_roi = 'HPC'

# Load all results if not already loaded
if 'all_group_results' not in dir() or len(all_group_results) == 0:
    all_group_results = {}
    for feedback_roi in feedback_rois:
        load_path = f'{save_dir}/group_results_{feedback_roi}_to_{cue_roi}.pkl'
        if os.path.exists(load_path):
            with open(load_path, 'rb') as f:
                all_group_results[feedback_roi] = pickle.load(f)
            print(f"Loaded: {load_path}")
        else:
            print(f"Not found: {load_path}")

# Store cluster test results for each ROI
all_cluster_results = {}

for feedback_roi in feedback_rois:
    if feedback_roi not in all_group_results:
        print(f"\nSkipping {feedback_roi} - no results available")
        continue
        
    group_results = all_group_results[feedback_roi]
    
    print(f"\n{'='*70}")
    print(f"CLUSTER TEST: {feedback_roi} → {cue_roi}")
    print(f"{'='*70}")
    
    # Get the stacked observed differences: (n_subjects, n_feedback_times, n_cue_times)
    X = group_results['observed_diffs_stack']
    
    print(f"Data shape for cluster test: {X.shape}")
    print(f"  (n_subjects={X.shape[0]}, n_feedback_times={X.shape[1]}, n_cue_times={X.shape[2]})")
    
    # Create adjacency matrix for 2D grid (feedback_times × cue_times)
    n_fb = X.shape[1]
    n_cue = X.shape[2]
    
    # Create separate adjacency for each dimension, then combine
    fb_adjacency = mne.stats.combine_adjacency(n_fb)
    cue_adjacency = mne.stats.combine_adjacency(n_cue)
    
    # Combine into 2D adjacency using Kronecker product logic
    adjacency_2d = kron(eye(n_fb), cue_adjacency) + kron(fb_adjacency, eye(n_cue))
    adjacency_2d = csr_matrix(adjacency_2d)
    
    # Reshape X to (n_subjects, n_fb * n_cue)
    X_flat = X.reshape(X.shape[0], -1)
    
    # Run cluster permutation test
    print("Running permutation cluster test...")
    
    t_obs, clusters, cluster_pv, H0 = permutation_cluster_1samp_test(
        X_flat,
        n_permutations=1000,
        threshold=None,
        adjacency=adjacency_2d,
        tail=1,
        n_jobs=8,
        seed=42,
        verbose=False
    )
    
    # Reshape t_obs back to 2D
    t_obs_2d = t_obs.reshape(n_fb, n_cue)
    
    # Find significant clusters
    sig_clusters = [clusters[i] for i in range(len(clusters)) if cluster_pv[i] < 0.05]
    
    print(f"Clusters found: {len(clusters)} total, {len(sig_clusters)} significant (p < 0.05)")
    
    for i, (cluster, pval) in enumerate(zip(clusters, cluster_pv)):
        if pval < 0.05:
            print(f"  Cluster {i+1}: size={len(cluster[0])}, p={pval:.4f}")
    
    # Store results
    all_cluster_results[feedback_roi] = {
        't_obs_2d': t_obs_2d,
        'clusters': clusters,
        'cluster_pv': cluster_pv,
        'sig_clusters': sig_clusters,
        'n_fb': n_fb,
        'n_cue': n_cue
    }

print(f"\n{'='*70}")
print(f"CLUSTER TESTS COMPLETE")
print(f"{'='*70}")


In [ ]:
X

In [ ]:
# =============================================================================
# Plot Results with Significant Clusters - ALL ROIs
# =============================================================================

save_dir_plot = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/HFA_RPE_RSA'

for feedback_roi in feedback_rois:
    if feedback_roi not in all_group_results or feedback_roi not in all_cluster_results:
        print(f"\nSkipping {feedback_roi} - no results available")
        continue
    
    group_results = all_group_results[feedback_roi]
    cluster_results = all_cluster_results[feedback_roi]
    
    t_obs_2d = cluster_results['t_obs_2d']
    clusters = cluster_results['clusters']
    cluster_pv = cluster_results['cluster_pv']
    sig_clusters = cluster_results['sig_clusters']
    n_fb = cluster_results['n_fb']
    n_cue = cluster_results['n_cue']
    
    print(f"\n{'='*70}")
    print(f"PLOTTING: {feedback_roi} → {cue_roi}")
    print(f"{'='*70}")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    feedback_times = group_results['feedback_times']
    cue_times = group_results['cue_times']
    
    # 1. T-statistics from cluster test with cluster outlines
    ax = axes[0]
    t_max = np.nanmax(np.abs(t_obs_2d))
    im = ax.imshow(t_obs_2d, aspect='auto', origin='lower',
                   extent=[cue_times[0], cue_times[-1], feedback_times[0], feedback_times[-1]],
                   cmap='RdBu_r', vmin=-4, vmax=4)
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    
    # Overlay significant clusters
    if len(sig_clusters) > 0:
        # Create mask for significant clusters
        sig_mask = np.zeros(n_fb * n_cue, dtype=bool)
        for cluster in sig_clusters:
            sig_mask[cluster[0]] = True
        sig_mask_2d = sig_mask.reshape(n_fb, n_cue)
        
        # Draw contour around significant clusters
        ax.contour(cue_times, feedback_times, sig_mask_2d.astype(float),
                   levels=[0.5], colors='black', linewidths=2)
    
    ax.set_title(f'T-statistics (Cluster Permutation Test)\n{len(sig_clusters)} significant clusters (p < 0.05)')
    ax.set_xlabel('Cue/Retrieval Time (s)')
    ax.set_ylabel('Feedback/Encoding Time (s)')
    plt.colorbar(im, ax=ax, label='T-statistic')
    
    # 2. Group mean with significant clusters highlighted
    ax = axes[1]
    vmax = np.nanmax(np.abs(group_results['group_observed_diff']))
    im = ax.imshow(group_results['group_observed_diff'], aspect='auto', origin='lower',
                   extent=[cue_times[0], cue_times[-1], feedback_times[0], feedback_times[-1]],
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    
    # Overlay significant clusters
    if len(sig_clusters) > 0:
        ax.contour(cue_times, feedback_times, sig_mask_2d.astype(float),
                   levels=[0.5], colors='black', linewidths=2)
    
    ax.set_title(f'Group Mean RSA Difference\n(Remembered - Forgotten)')
    ax.set_xlabel('Cue/Retrieval Time (s)')
    ax.set_ylabel('Feedback/Encoding Time (s)')
    plt.colorbar(im, ax=ax, label='Mean ΔRSA')
    
    plt.suptitle(f"Cross-Temporal RSA: {group_results['feedback_roi']} → {group_results['cue_roi']}\n"
                 f"N={group_results['n_subjects']} subjects | Cluster test: {len(clusters)} clusters, "
                 f"{len(sig_clusters)} significant",
                 fontsize=12)
    plt.tight_layout()
    
    # Save plot
    save_path = f'{save_dir_plot}/cross_temporal_rsa_cluster_{feedback_roi}_{cue_roi}.pdf'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {save_path}")
    plt.show()
    
    # Print cluster details
    print(f"\nCluster Details for {feedback_roi} → {cue_roi}:")
    print("-" * 50)
    for i, (cluster, pval) in enumerate(sorted(zip(clusters, cluster_pv), key=lambda x: x[1])):
        cluster_mask = np.zeros(n_fb * n_cue, dtype=bool)
        cluster_mask[cluster[0]] = True
        cluster_2d = cluster_mask.reshape(n_fb, n_cue)
        
        # Find the time ranges of the cluster
        fb_indices = np.where(cluster_2d.any(axis=1))[0]
        cue_indices = np.where(cluster_2d.any(axis=0))[0]
        
        fb_range = (feedback_times[fb_indices[0]], feedback_times[fb_indices[-1]])
        cue_range = (cue_times[cue_indices[0]], cue_times[cue_indices[-1]])
        
        # Get mean effect in cluster
        mean_effect = np.mean(group_results['group_observed_diff'][cluster_2d])
        
        sig_marker = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"Cluster {i+1}: p={pval:.4f} {sig_marker}")
        print(f"  Size: {len(cluster[0])} time-points")
        print(f"  Feedback time: {fb_range[0]:.2f} to {fb_range[1]:.2f} s")
        print(f"  Cue time: {cue_range[0]:.2f} to {cue_range[1]:.2f} s")
        print(f"  Mean effect: {mean_effect:.4f}")

print(f"\n{'='*70}")
print("ALL PLOTS COMPLETE")
print(f"{'='*70}")


In [ ]:
# =============================================================================
# SCHEMATIC FIGURE UTILITIES
# =============================================================================
# Helper functions to generate clean, exportable plots for manuscript schematics.
# These produce minimal-style figures suitable for import into vector graphics software.


In [ ]:
# =============================================================================
# Helper Functions for Schematic-Ready Plots
# =============================================================================

def compute_rdm_at_timepoint(data, timepoint_idx, epoch='feedback', memory_condition=None):
    """
    Compute and return an RDM at a specific timepoint for visualization.
    
    Parameters
    ----------
    data : dict
        Preprocessed data dictionary from load_subject_data()
    timepoint_idx : int
        Index of the timepoint to extract
    epoch : str
        'feedback' or 'cue' - which epoch to extract from
    memory_condition : str or None
        'remembered', 'forgotten', or None (all trials)
    
    Returns
    -------
    rdm : ndarray
        The dissimilarity matrix (n_trials x n_trials)
    """
    if epoch == 'feedback':
        hfa = data['feedback_hfa']
    else:
        hfa = data['cue_hfa']
    
    # Extract patterns at this timepoint: (n_trials, n_electrodes)
    patterns = hfa[:, :, timepoint_idx]
    
    # Filter by memory condition if specified
    if memory_condition is not None:
        mask = data['memory_outcomes'] == memory_condition
        patterns = patterns[mask]
    
    # Compute RDM using correlation distance
    rdm = squareform(pdist(patterns, metric='correlation'))
    return rdm


def plot_example_rdm(rdm, ax=None, cmap='viridis', show_colorbar=False, 
                     show_labels=False, title=None, vmin=None, vmax=None):
    """
    Plot a single RDM with minimal styling for schematic use.
    
    Parameters
    ----------
    rdm : ndarray
        Dissimilarity matrix to plot
    ax : matplotlib axis or None
        Axis to plot on (creates new figure if None)
    cmap : str
        Colormap name
    show_colorbar : bool
        Whether to show colorbar
    show_labels : bool
        Whether to show axis labels
    title : str or None
        Optional title
    vmin, vmax : float or None
        Color scale limits
    
    Returns
    -------
    fig, ax : matplotlib figure and axis
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
    else:
        fig = ax.figure
    
    im = ax.imshow(rdm, cmap=cmap, aspect='equal', vmin=vmin, vmax=vmax)
    
    if not show_labels:
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        ax.set_xlabel('Trials')
        ax.set_ylabel('Trials')
    
    if title:
        ax.set_title(title, fontsize=10)
    
    if show_colorbar:
        plt.colorbar(im, ax=ax, shrink=0.8, label='Dissimilarity')
    
    # Clean border
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    
    return fig, ax


def plot_pattern_vectors(data, trial_indices, timepoint_idx, epoch='feedback',
                         ax=None, cmap='RdBu_r', show_labels=False):
    """
    Visualize HFA pattern vectors for specific trials as a heatmap.
    
    Parameters
    ----------
    data : dict
        Preprocessed data dictionary from load_subject_data()
    trial_indices : list or array
        Which trials to show
    timepoint_idx : int
        Timepoint to extract
    epoch : str
        'feedback' or 'cue'
    ax : matplotlib axis or None
    cmap : str
        Colormap
    show_labels : bool
        Whether to show axis labels
    
    Returns
    -------
    fig, ax : matplotlib figure and axis
    """
    if epoch == 'feedback':
        hfa = data['feedback_hfa']
    else:
        hfa = data['cue_hfa']
    
    # Extract patterns: (n_selected_trials, n_electrodes)
    patterns = hfa[trial_indices, :, timepoint_idx]
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 3))
    else:
        fig = ax.figure
    
    vmax = np.max(np.abs(patterns))
    im = ax.imshow(patterns, cmap=cmap, aspect='auto', vmin=-vmax, vmax=vmax)
    
    if show_labels:
        ax.set_xlabel('Electrodes')
        ax.set_ylabel('Trials')
    else:
        ax.set_xticks([])
        ax.set_yticks([])
    
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    
    return fig, ax


def plot_single_subject_heatmap(result, condition='diff', ax=None, cmap='RdBu_r',
                                 show_colorbar=False, show_labels=False, 
                                 show_zero_lines=True, title=None):
    """
    Plot a clean single-subject cross-temporal RSA heatmap.
    
    Parameters
    ----------
    result : dict
        Result from analyze_subject_rsa() or the 'observed' key from run_permutation_rsa()
    condition : str
        'remembered', 'forgotten', or 'diff' (remembered - forgotten)
    ax : matplotlib axis or None
    cmap : str
        Colormap
    show_colorbar : bool
        Whether to show colorbar
    show_labels : bool
        Whether to show axis labels
    show_zero_lines : bool
        Whether to show dashed lines at t=0
    title : str or None
        Optional title
    
    Returns
    -------
    fig, ax : matplotlib figure and axis
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    else:
        fig = ax.figure
    
    if condition == 'diff':
        heatmap = result['diff_heatmap']
    else:
        heatmap = result['category_heatmaps'][condition]
    
    feedback_times = result['feedback_times']
    cue_times = result['cue_times']
    
    vmax = np.nanmax(np.abs(heatmap))
    im = ax.imshow(heatmap, aspect='auto', origin='lower',
                   extent=[cue_times[0], cue_times[-1], 
                          feedback_times[0], feedback_times[-1]],
                   cmap=cmap, vmin=-vmax, vmax=vmax)
    
    if show_zero_lines:
        ax.axhline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
        ax.axvline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    
    if show_labels:
        ax.set_xlabel('Retrieval Time (s)')
        ax.set_ylabel('Encoding Time (s)')
    else:
        ax.set_xticks([])
        ax.set_yticks([])
    
    if title:
        ax.set_title(title, fontsize=10)
    
    if show_colorbar:
        plt.colorbar(im, ax=ax, shrink=0.8)
    
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    
    return fig, ax


def plot_time_slice(result, fixed_time, fixed_axis='encoding', ax=None,
                    show_legend=True, show_labels=False, colors=None):
    """
    Plot a 1D slice through the cross-temporal heatmap.
    
    Parameters
    ----------
    result : dict
        Result from analyze_subject_rsa()
    fixed_time : float
        Time (in seconds) to fix
    fixed_axis : str
        'encoding' (fix feedback time, vary cue time) or 
        'retrieval' (fix cue time, vary feedback time)
    ax : matplotlib axis or None
    show_legend : bool
        Whether to show legend
    show_labels : bool
        Whether to show axis labels
    colors : dict or None
        Colors for remembered/forgotten lines
    
    Returns
    -------
    fig, ax : matplotlib figure and axis
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 3))
    else:
        fig = ax.figure
    
    if colors is None:
        colors = {'remembered': '#2ecc71', 'forgotten': '#e74c3c'}
    
    feedback_times = result['feedback_times']
    cue_times = result['cue_times']
    
    if fixed_axis == 'encoding':
        # Find nearest feedback time index
        time_idx = np.argmin(np.abs(feedback_times - fixed_time))
        x_vals = cue_times
        x_label = 'Retrieval Time (s)'
    else:
        # Find nearest cue time index
        time_idx = np.argmin(np.abs(cue_times - fixed_time))
        x_vals = feedback_times
        x_label = 'Encoding Time (s)'
    
    for condition in ['remembered', 'forgotten']:
        heatmap = result['category_heatmaps'][condition]
        if fixed_axis == 'encoding':
            y_vals = heatmap[time_idx, :]
        else:
            y_vals = heatmap[:, time_idx]
        
        ax.plot(x_vals, y_vals, color=colors[condition], 
                label=condition.capitalize(), linewidth=1.5)
    
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    
    if show_labels:
        ax.set_xlabel(x_label)
        ax.set_ylabel('RSA Correlation')
    
    if show_legend:
        ax.legend(frameon=False, fontsize=8)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    return fig, ax


def export_schematic_figure(fig, filename, formats=['pdf', 'svg'], dpi=300):
    """
    Export figure in multiple formats for schematic use.
    
    Parameters
    ----------
    fig : matplotlib figure
        Figure to export
    filename : str
        Base filename (without extension)
    formats : list
        List of formats to export (e.g., ['pdf', 'svg', 'png'])
    dpi : int
        DPI for raster formats
    """
    for fmt in formats:
        fig.savefig(f'{filename}.{fmt}', format=fmt, dpi=dpi, 
                    bbox_inches='tight', transparent=True)
        print(f"Saved: {filename}.{fmt}")


In [ ]:
# =============================================================================
# EXAMPLE USAGE: Generate Schematic Components
# =============================================================================
# Run this cell after loading data for a subject to generate exportable plots.
# Adjust parameters as needed for your schematic design.

# --- Find subject with most HPC electrodes that also has ACC coverage ---
feedback_roi_schematic = 'ACC'
cue_roi_schematic = 'HPC'

electrode_counts = []
for subj in subj_df.MSSMCode.values:
    elec_file = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/Day1_reref_elec_df'
    if os.path.exists(elec_file):
        elec_df = pd.read_csv(elec_file)
        n_hpc = len(elec_df[elec_df.salman_region == cue_roi_schematic])
        n_acc = len(elec_df[elec_df.salman_region == feedback_roi_schematic])
        if n_hpc > 0 and n_acc > 0:
            electrode_counts.append({'subject': subj, 'n_hpc': n_hpc, 'n_acc': n_acc})

electrode_counts_df = pd.DataFrame(electrode_counts)
print("Subjects with both ACC and HPC coverage:")
print(electrode_counts_df.sort_values('n_hpc', ascending=False).to_string(index=False))

# Select subject with most HPC electrodes
example_subject = electrode_counts_df.sort_values('n_hpc', ascending=False).iloc[0]['subject']
print(f"\nSelected example subject: {example_subject}")

# --- Load data for selected subject ---
example_data = load_subject_data(
    subject=example_subject,
    base_dir=base_dir,
    load_dir=load_dir,
    feedback_roi=feedback_roi_schematic,
    cue_roi=cue_roi_schematic,
    downsample_factor=5,
    verbose=True
)

# Compute RSA result for this subject
example_result = analyze_subject_rsa(
    subject=example_subject,
    base_dir=base_dir,
    load_dir=load_dir,
    feedback_roi=feedback_roi_schematic,
    cue_roi=cue_roi_schematic,
    downsample_factor=5,
    verbose=True
)


In [ ]:
# =============================================================================
# EXAMPLE 1: Plot RDMs at specific timepoints
# =============================================================================
# These can be used to show "what an RDM looks like" in your schematic
# RDM = Representational Dissimilarity Matrix: pairwise distances between trial patterns

if example_data is not None:
    # Find timepoint indices for key moments
    feedback_times = example_data['feedback_times']
    cue_times = example_data['cue_times']
    
    # Pick timepoints of interest
    fb_time_of_interest = 0.8  # seconds after feedback
    cue_time_of_interest = 0.5  # seconds after cue
    
    fb_idx = np.argmin(np.abs(feedback_times - fb_time_of_interest))
    cue_idx = np.argmin(np.abs(cue_times - cue_time_of_interest))
    
    print(f"Feedback timepoint: {feedback_times[fb_idx]:.2f}s (index {fb_idx})")
    print(f"Cue timepoint: {cue_times[cue_idx]:.2f}s (index {cue_idx})")
    
    # Compute RDMs for remembered and forgotten trials
    rdm_rem_fb = compute_rdm_at_timepoint(example_data, fb_idx, 'feedback', 'remembered')
    rdm_forg_fb = compute_rdm_at_timepoint(example_data, fb_idx, 'feedback', 'forgotten')
    rdm_rem_cue = compute_rdm_at_timepoint(example_data, cue_idx, 'cue', 'remembered')
    rdm_forg_cue = compute_rdm_at_timepoint(example_data, cue_idx, 'cue', 'forgotten')
    
    # Plot RDMs with proper annotations
    fig, axes = plt.subplots(2, 2, figsize=(10, 9))
    
    # Use consistent color scale
    vmax = max(np.nanmax(rdm_rem_fb), np.nanmax(rdm_forg_fb), 
               np.nanmax(rdm_rem_cue), np.nanmax(rdm_forg_cue))
    
    # Row 1: Encoding (feedback) RDMs
    plot_example_rdm(rdm_rem_fb, ax=axes[0, 0], vmax=vmax, show_labels=True)
    axes[0, 0].set_title(f'Remembered Trials\n(n={rdm_rem_fb.shape[0]})', fontsize=11)
    
    plot_example_rdm(rdm_forg_fb, ax=axes[0, 1], vmax=vmax, show_labels=True)
    axes[0, 1].set_title(f'Forgotten Trials\n(n={rdm_forg_fb.shape[0]})', fontsize=11)
    
    # Row 2: Retrieval (cue) RDMs
    plot_example_rdm(rdm_rem_cue, ax=axes[1, 0], vmax=vmax, show_labels=True)
    axes[1, 0].set_title(f'Remembered Trials\n(n={rdm_rem_cue.shape[0]})', fontsize=11)
    
    im = axes[1, 1].imshow(rdm_forg_cue, cmap='viridis', aspect='equal', vmax=vmax)
    axes[1, 1].set_title(f'Forgotten Trials\n(n={rdm_forg_cue.shape[0]})', fontsize=11)
    axes[1, 1].set_xlabel('Trials')
    axes[1, 1].set_ylabel('Trials')
    
    # Add row labels on the left
    axes[0, 0].set_ylabel(f'Encoding ({feedback_roi_schematic})\nt = {feedback_times[fb_idx]:.2f}s\n\nTrials', fontsize=10)
    axes[1, 0].set_ylabel(f'Retrieval ({cue_roi_schematic})\nt = {cue_times[cue_idx]:.2f}s\n\nTrials', fontsize=10)
    
    # Add colorbar
    cbar = fig.colorbar(im, ax=axes, shrink=0.6, label='Dissimilarity\n(correlation distance)')
    
    # Main title
    fig.suptitle(f'Representational Dissimilarity Matrices (RDMs)\nSubject: {example_subject}', 
                 fontsize=13, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    
    # Uncomment to export:
    export_schematic_figure(fig, f'{base_dir}/scratch/MemoryBandit/RSA/schematic_rdms')


In [ ]:
# =============================================================================
# EXAMPLE 2: Plot pattern vectors (what goes into the RDM)
# =============================================================================
# Shows HFA (high-frequency activity) patterns across electrodes for a subset of trials
# Each row = one trial, each column = one electrode
# These patterns are what get compared to compute the RDM

if example_data is not None:
    # Select a few example trials
    n_example_trials = 10
    trial_indices = np.arange(min(n_example_trials, example_data['n_trials']))
    
    # Pick a timepoint
    timepoint_idx = fb_idx
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Get number of electrodes for labels
    n_fb_elec = example_data['feedback_hfa'].shape[1]
    n_cue_elec = example_data['cue_hfa'].shape[1]
    
    # Encoding patterns
    plot_pattern_vectors(example_data, trial_indices, timepoint_idx, 
                         epoch='feedback', ax=axes[0], show_labels=True)
    axes[0].set_title(f'{feedback_roi_schematic} Activity During Encoding\n(t = {feedback_times[timepoint_idx]:.2f}s post-feedback)', 
                      fontsize=11)
    axes[0].set_xlabel(f'{feedback_roi_schematic} Electrodes (n={n_fb_elec})')
    axes[0].set_ylabel('Trials')
    
    # Add colorbar for encoding
    im0 = axes[0].images[0]
    cbar0 = fig.colorbar(im0, ax=axes[0], shrink=0.8, pad=0.02)
    cbar0.set_label('HFA\n(z-scored)', fontsize=9)
    
    # Retrieval patterns
    plot_pattern_vectors(example_data, trial_indices, cue_idx, 
                         epoch='cue', ax=axes[1], show_labels=True)
    axes[1].set_title(f'{cue_roi_schematic} Activity During Retrieval\n(t = {cue_times[cue_idx]:.2f}s post-cue)', 
                      fontsize=11)
    axes[1].set_xlabel(f'{cue_roi_schematic} Electrodes (n={n_cue_elec})')
    axes[1].set_ylabel('Trials')
    
    # Add colorbar for retrieval
    im1 = axes[1].images[0]
    cbar1 = fig.colorbar(im1, ax=axes[1], shrink=0.8, pad=0.02)
    cbar1.set_label('HFA\n(z-scored)', fontsize=9)
    
    fig.suptitle(f'Neural Pattern Vectors Used for RSA\nSubject: {example_subject}', 
                 fontsize=13, fontweight='bold', y=1.05)
    
    plt.tight_layout()
    
    # Uncomment to export:
    # export_schematic_figure(fig, f'{base_dir}/scratch/MemoryBandit/RSA/schematic_patterns')


In [ ]:
# =============================================================================
# EXAMPLE 3: Single-subject cross-temporal heatmaps
# =============================================================================
# Cross-temporal RSA: correlating RDMs between encoding and retrieval timepoints
# X-axis: time during retrieval (when cue appears)
# Y-axis: time during encoding (when feedback appears)
# Color: correlation between RDMs at those two timepoints

if example_result is not None:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    
    # Get trial counts for labels
    n_rem = example_result['n_trials_per_category']['remembered']
    n_forg = example_result['n_trials_per_category']['forgotten']
    
    # Remembered trials
    plot_single_subject_heatmap(example_result, condition='remembered', 
                                 ax=axes[0], show_labels=True)
    axes[0].set_title(f'Remembered Trials (n={n_rem})', fontsize=11, fontweight='bold')
    axes[0].set_xlabel(f'Retrieval Time ({cue_roi_schematic}, s)')
    axes[0].set_ylabel(f'Encoding Time ({feedback_roi_schematic}, s)')
    im0 = axes[0].images[0]
    cbar0 = fig.colorbar(im0, ax=axes[0], shrink=0.8, pad=0.02)
    cbar0.set_label('RDM\nCorrelation', fontsize=9)
    
    # Forgotten trials
    plot_single_subject_heatmap(example_result, condition='forgotten', 
                                 ax=axes[1], show_labels=True)
    axes[1].set_title(f'Forgotten Trials (n={n_forg})', fontsize=11, fontweight='bold')
    axes[1].set_xlabel(f'Retrieval Time ({cue_roi_schematic}, s)')
    axes[1].set_ylabel(f'Encoding Time ({feedback_roi_schematic}, s)')
    im1 = axes[1].images[0]
    cbar1 = fig.colorbar(im1, ax=axes[1], shrink=0.8, pad=0.02)
    cbar1.set_label('RDM\nCorrelation', fontsize=9)
    
    # Difference (key result)
    plot_single_subject_heatmap(example_result, condition='diff', 
                                 ax=axes[2], show_labels=True)
    axes[2].set_title('Difference\n(Remembered − Forgotten)', fontsize=11, fontweight='bold')
    axes[2].set_xlabel(f'Retrieval Time ({cue_roi_schematic}, s)')
    axes[2].set_ylabel(f'Encoding Time ({feedback_roi_schematic}, s)')
    im2 = axes[2].images[0]
    cbar2 = fig.colorbar(im2, ax=axes[2], shrink=0.8, pad=0.02)
    cbar2.set_label('ΔRSA\n(Rem−Forg)', fontsize=9)
    
    fig.suptitle(f'Cross-Temporal RSA: {feedback_roi_schematic} → {cue_roi_schematic}\n'
                 f'Subject: {example_subject} | Higher values = more similar encoding-retrieval patterns', 
                 fontsize=12, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    
    # Uncomment to export:
    # export_schematic_figure(fig, f'{base_dir}/scratch/MemoryBandit/RSA/schematic_heatmaps')


In [ ]:
# =============================================================================
# EXAMPLE 4: Time-slice plots (1D cross-sections through the heatmap)
# =============================================================================
# These are 1D "slices" through the 2D cross-temporal heatmap
# Useful for showing temporal dynamics at a specific fixed timepoint

if example_result is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Define colors with good contrast
    mem_colors = {'remembered': '#2ecc71', 'forgotten': '#e74c3c'}
    
    # Left panel: Fix encoding time at 0.8s, show how RSA varies across retrieval time
    enc_time = 0.8
    plot_time_slice(example_result, fixed_time=enc_time, fixed_axis='encoding', 
                    ax=axes[0], show_labels=True, colors=mem_colors)
    axes[0].set_title(f'RDM Correlation Across Retrieval Time\n(Encoding fixed at t={enc_time}s in {feedback_roi_schematic})', 
                      fontsize=11)
    axes[0].set_xlabel(f'Retrieval Time in {cue_roi_schematic} (s)')
    axes[0].set_ylabel('RDM Correlation (Spearman ρ)')
    axes[0].legend(loc='upper right', frameon=True, fancybox=True, framealpha=0.9)
    axes[0].axvline(0, color='gray', linestyle=':', linewidth=1, alpha=0.7, label='Cue onset')
    
    # Right panel: Fix retrieval time at 0.5s, show how RSA varies across encoding time  
    ret_time = 0.5
    plot_time_slice(example_result, fixed_time=ret_time, fixed_axis='retrieval', 
                    ax=axes[1], show_labels=True, colors=mem_colors)
    axes[1].set_title(f'RDM Correlation Across Encoding Time\n(Retrieval fixed at t={ret_time}s in {cue_roi_schematic})', 
                      fontsize=11)
    axes[1].set_xlabel(f'Encoding Time in {feedback_roi_schematic} (s)')
    axes[1].set_ylabel('RDM Correlation (Spearman ρ)')
    axes[1].legend(loc='upper right', frameon=True, fancybox=True, framealpha=0.9)
    axes[1].axvline(0, color='gray', linestyle=':', linewidth=1, alpha=0.7, label='Feedback onset')
    
    fig.suptitle(f'Cross-Temporal RSA Time Slices | Subject: {example_subject}', 
                 fontsize=12, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    
    # Uncomment to export:
    # export_schematic_figure(fig, f'{base_dir}/scratch/MemoryBandit/RSA/schematic_timeslice')


In [ ]:
# =============================================================================
# EXAMPLE 5: Ultra-minimal exports for schematic composition
# =============================================================================
# These produce CLEAN figures with NO labels/titles/colorbars
# Intended for import into Illustrator/Inkscape where you add annotations manually
# Each figure is a single visual element you can use as a building block

if example_data is not None and example_result is not None:
    save_dir = f'{base_dir}/scratch/MemoryBandit/RSA/schematic_components'
    os.makedirs(save_dir, exist_ok=True)
    
    print("=" * 60)
    print("MINIMAL EXPORTS FOR SCHEMATIC COMPOSITION")
    print("=" * 60)
    
    # --- 1. Minimal RDM (single, all trials) ---
    print("\n1. RDM (Representational Dissimilarity Matrix)")
    print("   - Square matrix showing pairwise trial dissimilarity")
    print("   - Each cell = correlation distance between two trials")
    fig, ax = plt.subplots(figsize=(3, 3))
    rdm = compute_rdm_at_timepoint(example_data, fb_idx, 'feedback', None)
    plot_example_rdm(rdm, ax=ax, show_labels=False, show_colorbar=False)
    ax.set_title('RDM', fontsize=10)  # Minimal title for reference
    plt.tight_layout()
    # export_schematic_figure(fig, f'{save_dir}/rdm_single')
    plt.show()
    
    # --- 2. Minimal cross-temporal heatmap (difference) ---
    print("\n2. Cross-Temporal RSA Heatmap (Remembered - Forgotten)")
    print("   - X-axis: retrieval time, Y-axis: encoding time")
    print("   - Color: difference in RDM correlation between memory conditions")
    fig, ax = plt.subplots(figsize=(4, 3))
    plot_single_subject_heatmap(example_result, condition='diff', ax=ax,
                                 show_labels=False, show_colorbar=False, 
                                 show_zero_lines=False)
    ax.set_title('Cross-Temporal RSA', fontsize=10)
    plt.tight_layout()
    # export_schematic_figure(fig, f'{save_dir}/heatmap_diff')
    plt.show()
    
    # --- 3. Minimal pattern vectors ---
    print("\n3. Pattern Vectors (HFA across electrodes)")
    print("   - Rows = trials, Columns = electrodes")
    print("   - Color = high-frequency activity (HFA)")
    fig, ax = plt.subplots(figsize=(4, 2))
    trial_indices = np.arange(min(8, example_data['n_trials']))
    plot_pattern_vectors(example_data, trial_indices, fb_idx, 
                         epoch='feedback', ax=ax, show_labels=False)
    ax.set_title('Pattern Vectors', fontsize=10)
    plt.tight_layout()
    # export_schematic_figure(fig, f'{save_dir}/patterns')
    plt.show()
    
    print("\n" + "=" * 60)
    print("To export these as PDF/SVG for Illustrator:")
    print("  1. Uncomment the export_schematic_figure() lines above")
    print("  2. Re-run this cell")
    print(f"  3. Files will be saved to: {save_dir}")
    print("=" * 60)
